# 3. Fire stations: set covering

Lecture: *Week 5 - Mixed Integer Programming*, PDF pages 14–20 (*Set Covering Problem - Fire Stations Example*).

**The problem.** Six cities need fire stations. Build as few as possible, so that every city has a station
within 15 minutes' drive (the table of driving times is on PDF page 14). $x_i = 1$ if a station is built in
city $i$. Reading the table, a city is covered by a station in these cities:

| City | Stations within 15 minutes |
| --- | --- |
| 1 | 1, 2 |
| 2 | 1, 2, 6 |
| 3 | 3, 4 |
| 4 | 3, 4, 5 |
| 5 | 4, 5, 6 |
| 6 | 2, 5, 6 |

minimise $\sum_{i=1}^{6} x_i$ subject to one "at least one station" constraint per city (PDF page 18),
$x_i \in \{0, 1\}$. In the code, cities 1–6 are `x[0]` … `x[5]`.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** **2 stations**, in cities 2 and 4 (`x[1] = 1`, `x[3] = 1`): the answer found by inspection
on PDF page 15.

*Original script: `5 Source Files/Python/firestation_lecture.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A *concrete model* is the container for the problem.

In [ ]:
# Create a concrete model
model = ConcreteModel()

One binary variable per city: build a station there (1) or not (0).

In [ ]:
# Define the binary variables: x[i] = 1 if a fire station is built in city i+1
model.x = Var(range(6), within=Binary)

The objective: the number of stations, minimised.

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(model.x[i] for i in range(6)), sense=minimize)

One constraint per city: at least one of the stations within 15 minutes of it must be built (the table above).

In [ ]:
# Define the inequality constraints: every city has a station within 15 minutes
model.city1 = Constraint(expr=model.x[0] + model.x[1] >= 1)                # city 1: station in 1 or 2
model.city2 = Constraint(expr=model.x[0] + model.x[1] + model.x[5] >= 1)   # city 2: station in 1, 2 or 6
model.city3 = Constraint(expr=model.x[2] + model.x[3] >= 1)                # city 3: station in 3 or 4
model.city4 = Constraint(expr=model.x[2] + model.x[3] + model.x[4] >= 1)   # city 4: station in 3, 4 or 5
model.city5 = Constraint(expr=model.x[3] + model.x[4] + model.x[5] >= 1)   # city 5: station in 4, 5 or 6
model.city6 = Constraint(expr=model.x[1] + model.x[4] + model.x[5] >= 1)   # city 6: station in 2, 5 or 6

Create the HiGHS solver, print the model (`pprint()`: check it against the problem above), solve, and print the results (`display()`).

The HiGHS log appears between the two printouts (in the browser it is printed even without `tee=True`); its *Solving report* should say *Status Optimal*. In `display()`, look at the `Value` column of the variables and at the objective.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')
# Display the model
model.pprint()
# Solve the model
solver.solve(model)
# Display the results
model.display()

## What do you notice?

Is {2, 4} the only way to cover all six cities with two stations?

<details><summary>Answer</summary>

Yes. City 1 needs a station in 1 or 2, and city 3 needs one in 3 or 4, so two stations must be one from each
pair. {1, 3} misses cities 5 and 6, {1, 4} misses city 6, {2, 3} misses city 5; only {2, 4} covers all six.
Week 06 comes back to this example with building costs.
</details>